In [ ]:
! pip install yfinance

In [2]:
import pandas as pd
import numpy as np
import datetime as dt
import yfinance as yf
from scipy.stats import norm, t
import matplotlib.pyplot as plt

In [3]:
stockList = ['MSFT', 'AAPL', 'NVDA', 'AMZN', 'META', 'GOOGL']
stocks = [stock for stock in stockList]

In [ ]:
endDate = dt.datetime.now()
startDate = endDate - dt.timedelta(days=100)
stockData = yf.download(stocks, start=startDate, end=endDate)

In [5]:
stockData.head(2)

Price            Close                                                  \
Ticker            AAPL        AMZN       GOOGL        META        MSFT   
Date                                                                     
2026-02-17  263.637115  201.149994  301.807526  638.743103  395.100494   
2026-02-18  264.106689  204.789993  303.116608  642.669739  397.828369   

Price                         High                                      ...  \
Ticker            NVDA        AAPL        AMZN       GOOGL        META  ...   
Date                                                                    ...   
2026-02-17  184.959991  266.044901  201.740005  304.225837  642.050269  ...   
2026-02-18  187.969818  266.574417  206.860001  305.165184  644.448245  ...   

Price             Open                                        Volume  \
Ticker           GOOGL        META        MSFT        NVDA      AAPL   
Date                                                                   
2026-02-17  299.828938  638.952945  397.450047  181.740164  58469100   
2026-02-18  301.877490  633.257814  396.364885  188.739781  34203300   

Price                                                          
Ticker          AMZN     GOOGL      META      MSFT       NVDA  
Date                                                           
2026-02-17  69879200  39247600  12674700  32078800  162276900  
2026-02-18  51003300  28482100  14649200  23223400  164749100  

[2 rows x 30 columns]

In [6]:
stockData.swaplevel(axis=1).sort_index(axis=1).tail(2)

Ticker            AAPL                                                \
Price            Close        High         Low        Open    Volume   
Date                                                                   
2026-05-22  308.820007  311.399994  305.839996  306.119995  43670200   
2026-05-26  308.329987  311.820007  307.670013  309.399994  46602561   

Ticker            AMZN                                                ...  \
Price            Close        High         Low        Open    Volume  ...   
Date                                                                  ...   
2026-05-22  266.320007  269.790009  266.239990  268.660004  27535500  ...   
2026-05-26  265.290009  269.299988  262.070007  267.575012  37859240  ...   

Ticker            MSFT                                                \
Price            Close        High         Low        Open    Volume   
Date                                                                   
2026-05-22  418.570007  424.399994  416.329987  419.540009  22390300   
2026-05-26  416.029999  419.769989  413.019989  416.429993  30100943   

Ticker            NVDA                                                 
Price            Close        High         Low        Open     Volume  
Date                                                                   
2026-05-22  215.330002  221.009995  214.800003  220.899994  169275700  
2026-05-26  214.860001  218.179993  212.000000  216.577499  182422520  

[2 rows x 30 columns]

In [7]:
stockData['Close'].tail()

Ticker,AAPL,AMZN,GOOGL,META,MSFT,NVDA
Date,,,,,,
2026-05-19,298.970001,259.339996,387.660004,602.609985,416.517883,220.610001
2026-05-20,302.250000,265.010010,388.910004,605.059998,420.149994,223.470001
2026-05-21,304.989990,268.459991,387.660004,607.380005,419.089996,219.509995
2026-05-22,308.820007,266.320007,382.970001,610.260010,418.570007,215.330002
2026-05-26,308.329987,265.290009,388.880005,612.340027,416.029999,214.860001


$$\mathrm{Return}(t)=\frac{\mathrm{Price}(t)-\mathrm{Price}(t-1)}{\mathrm{Price}(t-1)}$$

In [34]:
returns = stockData['Close'].pct_change()
returns.tail()

Ticker,AAPL,AMZN,GOOGL,META,MSFT,NVDA
Date,,,,,,
2026-05-19,0.003794,-0.020841,-0.023379,-0.014071,-0.014450,-0.007692
2026-05-20,0.010971,0.021863,0.003224,0.004066,0.008720,0.012964
2026-05-21,0.009065,0.013018,-0.003214,0.003834,-0.002523,-0.017721
2026-05-22,0.012558,-0.007971,-0.012098,0.004742,-0.001241,-0.019042
2026-05-26,-0.001587,-0.003868,0.015432,0.003408,-0.006068,-0.002183


In [35]:
returns.shape

(69, 6)

La cantidad de registros corresponde a la cantidad de días quitando fines de semana y feriados.

In [44]:
mean_returns = returns.mean()
mean_returns

,0
Ticker,
AAPL,0.002398
AMZN,0.004233
GOOGL,0.003954
META,-0.000332
MSFT,0.000897
NVDA,0.002474


In [43]:
cov_matrix = returns.cov()
cov_matrix

Ticker,AAPL,AMZN,GOOGL,META,MSFT,NVDA
Ticker,,,,,,
AAPL,0.000188,0.000076,0.000096,0.000103,0.000062,0.000102
AMZN,0.000076,0.000317,0.000202,0.000267,0.000103,0.000177
GOOGL,0.000096,0.000202,0.000456,0.000148,0.000039,0.000143
META,0.000103,0.000267,0.000148,0.000578,0.000243,0.000356
MSFT,0.000062,0.000103,0.000039,0.000243,0.000279,0.000149
NVDA,0.000102,0.000177,0.000143,0.000356,0.000149,0.000544


La diagonal es la varianza de cada activo respectivamente. En finanzas, la varianza mide qué tan volátil es una acción por sí sola.

In [38]:
returns = returns.dropna()

In [39]:
# Generamos pesos aleatorios para el portafolio que sumen 1
weights = np.random.random(len(returns.columns))
weights = weights / np.sum(weights)
weights

array([0.22690238, 0.19367446, 0.05920083, 0.09226478, 0.23936699,
       0.18859055])

In [40]:
# Calculamos el rendimiento histórico del portafolio consolidado
returns.dot(weights).tail()

,0
Date,
2026-05-19,-0.010767
2026-05-20,0.011822
2026-05-21,0.000796
2026-05-22,-0.002861
2026-05-26,-0.001745


En la Teoría Moderna de Portafolios (de Harry Markowitz) el objetivo es proyectar en el futuro dos cosas fundamentales de un portafolio de inversión: cuánto dinero vas a ganar (Retorno) y cuánto riesgo estás asumiendo (Volatilidad/Desviación Estándar).

In [45]:
# Necesitamos los pesos de las acciones que compramos (weights), los rendimientos históricos promedio (mean_returns), 
# la matriz de covarianza (cov_matrix) y el horizonte de tiempo (time). 
# A partir de estos parámetros, calculamos el rendimiento esperado (expected_returns) y el riesgo del portafolio completo (std).

time = 100 
expected_returns = np.sum(mean_returns * weights) * time
std = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights))) * np.sqrt(time)

In [47]:
print("Rendimiento esperado del portafolio: ", expected_returns)
print("Riesgo del portafolio (desviación estándar): ", std)

Rendimiento esperado del portafolio:  0.22485299756324742
Riesgo del portafolio (desviación estándar):  0.13100633022308236


Al multiplicar el rendimiento promedio pasado por Time, estamos asumiendo matemáticamente el supuesto de que el futuro se va a comportar exactamente igual que el promedio del pasado. Aunque rendimientos pasados no garantizan rendimientos futuros. Es un "punto de partida" estándar: En estadística, ante la total imposibilidad de adivinar el futuro, el promedio histórico es matemáticamente el estimador menos sesgado que tenemos a la mano.

Si queremos que no sea una simple proyección lineal del pasado, se usan técnicas más avanzadas para simular un horizonte de tiempo. Simulaciones de Monte Carlo: En lugar de multiplicar por un horizonte de tiempo, usamos el promedio histórico y la covarianza como "reglas de juego" para que la computadora simule por ejemplo 10,000 futuros posibles diferentes día por día, añadiendo aleatoriedad (ruido estadístico). Al final, no obtenemos un solo número, sino una distribución de probabilidades _(ej. "Hay un 70% de probabilidad de que tu portafolio rinda entre tal y tal valor")_.

In [ ]:
# Parámetros globales de la simulación

InitialInvestment = 10000 # Inversión inicial en dólares
alpha = 5 # Significancia del 5% (equivalente a 95% de confianza)

In [ ]:
# ==========================================
# 1. OBTENCIÓN DE DATOS Y CONFIGURACIÓN
# ==========================================

def getData(stocks, start, end):
    # Descargamos los datos usando yfinance 
    stockData = yf.download(stocks, start=start, end=end)
    stockData = stockData['Close']
    
    # Calculamos rendimientos diarios logarítmicos/porcentuales
    returns = stockData.pct_change()
    meanReturns = returns.mean()
    covMatrix = returns.cov()
    return returns, meanReturns, covMatrix

def portfolioPerformance(weights, meanReturns, covMatrix, Time):
    returns = np.sum(meanReturns * weights) * Time
    std = np.sqrt(np.dot(weights.T, np.dot(covMatrix, weights))) * np.sqrt(Time)
    return returns, std

# Definimos los activos más populares del S&P 500 (Mercado de EE. UU.)
stockList = ['MSFT', 'AAPL', 'NVDA', 'AMZN', 'META', 'GOOGL']
stocks = [stock for stock in stockList]

# Definimos la ventana de tiempo (800 días hacia atrás)
endDate = dt.datetime.now()
startDate = endDate - dt.timedelta(days=800)

# Descarga y limpieza
returns, meanReturns, covMatrix = getData(stocks, start=startDate, end=endDate)
returns = returns.dropna()

# Generamos pesos aleatorios para el portafolio que sumen 1
weights = np.random.random(len(returns.columns))
weights /= np.sum(weights)

# Calculamos el rendimiento histórico del portafolio consolidado
returns['portfolio'] = returns.dot(weights)

# Parámetros globales de la simulación
Time = 100 # Horizonte de tiempo en días
InitialInvestment = 10000 # Inversión inicial en dólares
alpha = 5 # Significancia del 5% (equivalente a 95% de confianza)

# Rendimiento y desviación estándar esperados del portafolio en el horizonte de tiempo
pRet, pStd = portfolioPerformance(weights, meanReturns, covMatrix, Time)

In [ ]:
# ==========================================
# 2. MÉTODO HISTÓRICO
# ==========================================

def historicalVaR(returns, alpha=5):
    if isinstance(returns, pd.Series):
        return np.percentile(returns, alpha)
    elif isinstance(returns, pd.DataFrame):
        return returns.aggregate(historicalVaR, alpha=alpha)
    else:
        raise TypeError("Se esperaba un DataFrame o Series de Pandas")

def historicalCVaR(returns, alpha=5):
    if isinstance(returns, pd.Series):
        belowVaR = returns <= historicalVaR(returns, alpha=alpha)
        return returns[belowVaR].mean()
    elif isinstance(returns, pd.DataFrame):
        return returns.aggregate(historicalCVaR, alpha=alpha)
    else:
        raise TypeError("Se esperaba un DataFrame o Series de Pandas")

# Calculamos y escalamos los valores históricos
hVaR = -historicalVaR(returns['portfolio'], alpha=alpha) * np.sqrt(Time)
hCVaR = -historicalCVaR(returns['portfolio'], alpha=alpha) * np.sqrt(Time)

In [ ]:
# ==========================================
# 3. MÉTODO PARAMÉTRICO (Varianza-Covarianza)
# ==========================================

def var_parametric(portfolioReturns, portfolioStd, distribution='normal', alpha=5, dof=6):
    if distribution == 'normal':
        VaR = norm.ppf(1 - alpha/100) * portfolioStd - portfolioReturns
    elif distribution == 't-distribution':
        nu = dof
        VaR = np.sqrt((nu - 2) / nu) * t.ppf(1 - alpha/100, nu) * portfolioStd - portfolioReturns
    else:
        raise TypeError("Distribución no soportada. Use 'normal' o 't-distribution'")
    return VaR

def cvar_parametric(portfolioReturns, portfolioStd, distribution='normal', alpha=5, dof=6):
    if distribution == 'normal':
        CVaR = (alpha/100)**-1 * norm.pdf(norm.ppf(alpha/100)) * portfolioStd - portfolioReturns
    elif distribution == 't-distribution':
        nu = dof
        xanu = t.ppf(alpha/100, nu)
        CVaR = -1/(alpha/100) * (1 - nu)**(-1) * (nu - 2 + xanu**2) * t.pdf(xanu, nu) * portfolioStd - portfolioReturns
    else:
        raise TypeError("Distribución no soportada. Use 'normal' o 't-distribution'")
    return CVaR

# Cálculos Paramétricos
normVaR = var_parametric(pRet, pStd, distribution='normal', alpha=alpha)
normCVaR = cvar_parametric(pRet, pStd, distribution='normal', alpha=alpha)

tVaR = var_parametric(pRet, pStd, distribution='t-distribution', alpha=alpha, dof=6)
tCVaR = cvar_parametric(pRet, pStd, distribution='t-distribution', alpha=alpha, dof=6)

In [ ]:
# ==========================================
# 4. SIMULACIÓN DE MONTE CARLO
# ==========================================

mc_sims = 1000 # Incrementado a 1000 para mayor precisión matemática
T = Time

meanM = np.full(shape=(T, len(weights)), fill_value=meanReturns).T
portfolio_sims = np.full(shape=(T, mc_sims), fill_value=0.0)

# Simulación de trayectorias con correlación de Cholesky
for m in range(0, mc_sims):
    Z = np.random.normal(size=(T, len(weights)))
    L = np.linalg.cholesky(covMatrix)
    dailyReturns = meanM + np.inner(L, Z)
    portfolio_sims[:, m] = np.cumprod(np.inner(weights, dailyReturns.T) + 1) * InitialInvestment

# Funciones de riesgo para Monte Carlo
def mcVaR(returns, alpha=5):
    if isinstance(returns, pd.Series):
        return np.percentile(returns, alpha)
    else:
        raise TypeError("Se esperaba una serie de Pandas.")

def mcCVaR(returns, alpha=5):
    if isinstance(returns, pd.Series):
        belowVaR = returns <= mcVaR(returns, alpha=alpha)
        return returns[belowVaR].mean()
    else:
        raise TypeError("Se esperaba una serie de Pandas.")

# Extraemos los resultados del último día de la simulación
portResults = pd.Series(portfolio_sims[-1, :])

mc_VaR_val = InitialInvestment - mcVaR(portResults, alpha=alpha)
mc_CVaR_val = InitialInvestment - mcCVaR(portResults, alpha=alpha)

In [ ]:
# ==========================================
# 5. DESPLIEGUE DE RESULTADOS Y GRÁFICO
# ==========================================

print("-" * 50)
print(f"REPORTES DE RIESGO FINANCIERO (Inversión: ${InitialInvestment:,} a {Time} días)")
print("-" * 50)
print(f"Rendimiento Esperado del Portafolio : ${round(InitialInvestment * pRet, 2)}")
print("\n--- Métricas de Value at Risk (VaR 95%) ---")
print(f" Histórico VaR                      : ${round(InitialInvestment * hVaR, 2)}")
print(f" Paramétrico Normal VaR             : ${round(InitialInvestment * normVaR, 2)}")
print(f" Paramétrico t-Student VaR          : ${round(InitialInvestment * tVaR, 2)}")
print(f" Monte Carlo VaR                    : ${round(mc_VaR_val, 2)}")

print("\n--- Métricas de Conditional VaR (CVaR 95%) ---")
print(f" Histórico CVaR                     : ${round(InitialInvestment * hCVaR, 2)}")
print(f" Paramétrico Normal CVaR            : ${round(InitialInvestment * normCVaR, 2)}")
print(f" Paramétrico t-Student CVaR         : ${round(InitialInvestment * tCVaR, 2)}")
print(f" Monte Carlo CVaR                   : ${round(mc_CVaR_val, 2)}")
print("-" * 50)

# Generación del gráfico de Monte Carlo
plt.figure(figsize=(10, 6))
plt.plot(portfolio_sims, lw=0.5, alpha=0.6)
plt.axhline(InitialInvestment, color='black', linestyle='--', label='Inversión Inicial')
plt.ylabel('Valor del Portafolio ($)')
plt.xlabel('Días en el Futuro')
plt.title(f'Simulación de Monte Carlo ({mc_sims} escenarios a {T} días)')
plt.grid(True, alpha=0.3)
plt.show()